# b3_cells

> Phase 22 (`specs/multiagent/phase22/`, experiment B3): the fit's cells and targets. `FIT_CELLS` (15 clean cells, target `pooledShare`; 17 counting-board cells with independent liars, target `contamination`), `FOLDS` and `REUSED` (P22-3, P22-5); the grids (`COARSE`, `FINE_STEP`, `grid`) and θ's cache key (`thetaKey`, `parseKey`); `claudeRows`, `cellValues` and `claudeTargets` from the cached `runs.csv` rows (FR-B1). `b3_fit` imports them.

In [ ]:
#| default_exp b3_cells

In [ ]:
#| hide
from nbdev.showdoc import *

## Cells, folds and grids

A cell id is `<folder>:<level>:<N>`. θ is a dict of the knobs that are set; its key (`thetaKey`) names its cache file, `default` for θ_default. `gridSim` returns the cell means `{key: {cell id: mean}}`, which every later step reads.

In [ ]:
#| export
import csv
from pathlib import Path

import numpy as np

from explRL.b3_agents import B3_KNOBS

SIM_SEEDS = tuple(range(3000, 3030))                 # Part 0's seeds, as every Phase 18 forecast
_steps = lambda hi, d: tuple(float(x) for x in np.round(np.arange(0, hi + d / 2, d), 2))
COARSE = dict(trustOne=_steps(1, 0.2), spread=_steps(1, 0.2), shareAll=_steps(1, 0.2), guess=_steps(0.5, 0.1))
FINE_STEP = dict(trustOne=0.1, spread=0.1, shareAll=0.1, guess=0.05)
CANDIDATES = ('spread', 'shareAll', 'guess')          # behaviour 1's knob (P22-2); chooseKnob's ties go to 'spread'
S8, S5 = tuple(range(1800, 1808)), tuple(range(900, 905))


def _fc(folder, study, arm, fault, N, kind='colony', seeds=S8):
    lvl = arm if kind == 'p14' else f'{arm}|{fault}'
    return dict(id=f'{folder}:{lvl}:{N}', folder=folder, study=study, level=lvl, arm=arm, fault=fault, N=N, seeds=seeds,
                kind=kind, target='pooledShare' if fault == 'clean' else 'contamination')


_S4 = (('failFlag', 16), ('failFlag', 64), ('guardDeal', 16), ('guardDeal', 64), ('guardDealFlag', 16), ('squads', 16),
       ('squads', 64), ('squadsAll', 64))
FIT_CELLS = ([_fc('scale/full', 'scaleB', 'baseline', 'clean', N, 'p14', S5) for N in (8, 16, 32)] +
             [_fc('colony/stage1', 'colony1', a, 'clean', N) for a in ('plain', 'confirm', 'colony', 'colonyGuards') for N in (8, 16)] +
             [_fc('colony/stage32', 'colony32', a, 'clean', 32) for a in ('plain', 'colonyGuards')] +
             [_fc('colony/stage4', 'colony4', a, 'clean', 16) for a in ('failFlag', 'guardDeal')] +
             [_fc('colony/stage1', 'colony1', a, 'independent', N) for a in ('colony', 'colonyGuards') for N in (8, 16)] +
             [_fc('colony/stage32', 'colony32', 'colonyGuards', 'independent', 32),
              _fc('colony/stage64', 'colony64', 'colonyGuards', 'independent', 64)] +
             [_fc('colony/stage4', 'colony4', a, 'independent', N) for a, N in _S4] +
             [_fc('colony/stage4b', 'colony4b', a, 'independent', 32) for a in ('failFlag', 'guardDeal', 'squads')])
CELL = {c['id']: c for c in FIT_CELLS}
_ids = lambda f: tuple(c['id'] for c in FIT_CELLS if f(c))
FOLDS = dict(S1=_ids(lambda c: c['study'] == 'colony1'),
             S4=_ids(lambda c: c['study'] == 'colony4' or c['id'] in ('colony/stage64:colonyGuards|independent:64',
                                                                       'colony/stage1:colonyGuards|independent:16')),
             S4b=_ids(lambda c: c['study'] == 'colony4b' or c['id'] == 'colony/stage32:colonyGuards|independent:32'),
             P14=_ids(lambda c: c['kind'] == 'p14'))
REUSED = dict(C1=('colony/stage4:squads|independent:64', 'colony/stage64:colonyGuards|independent:64'))   # the 'all' fold drops these


def thetaKey(theta):
    "The canonical name of θ (a dict, or a key already): the set knobs in B3_KNOBS order, 'default' when none is set."
    if isinstance(theta, str):
        return theta
    return ','.join(f'{k}={float(theta[k]):.2f}' for k in B3_KNOBS if theta.get(k)) or 'default'


def parseKey(key):
    return {} if key == 'default' else {k: float(v) for k, v in (x.split('=') for x in key.split(','))}


def grid(knobs, steps=None):
    "Every θ over the product of each knob's values (COARSE, or 0..hi by FINE_STEP when steps='fine')."
    vals = [COARSE[k] if steps is None else _steps(B3_KNOBS[k][1], FINE_STEP[k]) for k in knobs]
    return [dict(zip(knobs, v)) for v in np.array(np.meshgrid(*vals, indexing='ij')).reshape(len(knobs), -1).T.tolist()]

In [ ]:
#| export
def claudeRows(root=None):
    "The cached Claude rows of every folder FIT_CELLS reads (NFR-2: rows only, never the response cache)."
    from explRL.bench_grid import readCSV
    from explRL.train import demo_artifact_dir
    root = Path(root or demo_artifact_dir(None, 'multiagent'))
    return [r for f in sorted({c['folder'] for c in FIT_CELLS}) for r in readCSV(root / f / 'runs.csv')]


def cellValues(rows, cells=FIT_CELLS):
    "{cell id: [per-board target]} over the cell's Claude boards (study, level, N and seeds match; runner llm)."
    f = lambda r, k: r.get(k) not in (None, '') and np.isfinite(float(r[k]))
    return {c['id']: [float(r[c['target']]) for r in rows if r.get('runner', 'llm') == 'llm' and r['study'] == c['study'] and
                      r['level'] == c['level'] and int(float(r['N'])) == c['N'] and int(float(r['seed'])) in c['seeds'] and
                      f(r, c['target'])] for c in cells}


def _target(v):
    return float(np.mean(v)), max(float(np.std(v, ddof=1) / np.sqrt(len(v))) if len(v) > 1 else 0.0, 0.01), len(v)


def claudeTargets(rows, cells=FIT_CELLS):
    "FR-B1: {cell id: (mean, se floored at 0.01, boards)}."
    return {k: _target(v) for k, v in cellValues(rows, cells).items() if v}

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()